<a href="https://colab.research.google.com/github/tadrossalama/zoafind/blob/master/bench.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install fastbook &> /dev/null
import os
from pathlib import Path
from fastai.collab import *
from fastai.vision.all import *
from fastbook import search_images_bing

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
path= Path('/content/drive/MyDrive/corals')

In [ ]:
coral_types = (
    'Acropora Cervicornis',
    'Acropora Palmata',
    'Catalaphyllia jardinae',
    'Dendrogyra cylindrus',
    'Diadema Antillarum',
    'Diploria Strigosa',
    'Euphyllia',
    'Elegance'
    'Gorgonians',
    'Heliofungia actiniformis',
    'Millepora Alcicornis',
    'Montastraea Cavernosa',
    'Meandrina Meandrites',
    'Montipora',
    'Palythoas Palythoa',
    'Siderastrea Siderea',
    'Tunicates',
)


In [ ]:
add_coral_types2 = (
    'Acanthastrea',
    'Alveopora',
    'Blastomussa',
    'Caulastraea Furcata',
    'Euphyllia Divisa',
    'Porites',
    'Pocillopora meandrina',
    'Hydnophora',
    'Cyphastrea',
    'Seriatopora',
    'Leptoseris',
    'Discosoma',
    'Rhodactis',
    'Ricordea',
    'Duncanopsammia axifuga',
    'Gorgoniidae',
    'Sinularia',
    'Xeniidae'
)

In [ ]:
for o in add_coral_types2:
    dest=(path/o)
    dest.mkdir(exist_ok=True)
    results = search_images_bing(key, f'{o}')
    download_images(dest, urls=results.attrgot('contentUrl'))

In [ ]:
def search_images_bing(key, term, min_sz=128, max_images=150):    
     params = {'q':term, 'count':max_images, 'min_height':min_sz, 'min_width':min_sz}
     headers = {"Ocp-Apim-Subscription-Key":key}
     search_url = "https://api.bing.microsoft.com/v7.0/images/search"
     response = requests.get(search_url, headers=headers, params=params)
     response.raise_for_status()
     search_results = response.json()    
     return L(search_results['value'])

In [5]:
key = os.environ.get('AZURE_SEARCH_KEY', '620cc1546a0c48ceb0f937e7e0b0c25a')

In [6]:
def add_species(name):
    dest=(path/name)
    dest.mkdir(exist_ok=True)
    results = search_images_bing(key, f'{name}')
    download_images(dest, urls=results.attrgot('contentUrl'))

add_species('Favia')

In [ ]:
fns= get_image_files(path)
failed = verify_images(fns)
failed.map(Path.unlink)

In [ ]:
batch_tfms = [RandomResizedCrop(224), *aug_transforms(mult=1.0, do_flip=True, max_rotate=45,
                            max_lighting=.8, max_warp=0.1, p_lighting=.5)]

In [ ]:
corals = DataBlock(blocks = (ImageBlock, CategoryBlock),
                 get_items=get_image_files,
                 splitter= RandomSplitter(seed=42),       #splits items between train/val randomly
                 get_y=parent_label,    
                 item_tfms=Resize(460),
                 batch_tfms=batch_tfms)

In [ ]:
dls= corals.dataloaders(path)


## Resnet50

 ### Fastai:
 An idea to reduce memory usage (and avoid those annoying cuda errors) has been to try and do the same thing in half-precision, which means using 16-bits floats (or FP16 in the rest of this post). By definition, they take half the space in RAM, and in theory could allow you to double the size of your model and double your batch size.

In [ ]:
learn3 = vision_learner(dls, models.resnet50, metrics=error_rate).to_fp16()
learn3.fit(3, cbs=BnFreeze)
learn3.fine_tune(6, freeze_epochs=3)

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


  0%|          | 0.00/97.8M [00:00<?, ?B/s]

epoch,train_loss,valid_loss,error_rate,time
0,2.091567,0.844426,0.234354,11:26
1,1.324977,0.718351,0.225033,01:26
2,0.980600,0.608623,0.181092,01:26


/usr/local/lib/python3.7/dist-packages/PIL/TiffImagePlugin.py:788: UserWarning: Corrupt EXIF data.  Expecting to read 4 bytes but only got 0. 
  warnings.warn(str(msg))
/usr/local/lib/python3.7/dist-packages/PIL/TiffImagePlugin.py:788: UserWarning: Corrupt EXIF data.  Expecting to read 4 bytes but only got 0. 
  warnings.warn(str(msg))
/usr/local/lib/python3.7/dist-packages/PIL/TiffImagePlugin.py:788: UserWarning: Corrupt EXIF data.  Expecting to read 4 bytes but only got 0. 
  warnings.warn(str(msg))


epoch,train_loss,valid_loss,error_rate,time
0,0.626505,0.562672,0.165113,01:27
1,0.661949,0.611404,0.178429,01:27
2,0.689007,0.621316,0.174434,01:26


/usr/local/lib/python3.7/dist-packages/PIL/TiffImagePlugin.py:788: UserWarning: Corrupt EXIF data.  Expecting to read 4 bytes but only got 0. 
  warnings.warn(str(msg))
/usr/local/lib/python3.7/dist-packages/PIL/TiffImagePlugin.py:788: UserWarning: Corrupt EXIF data.  Expecting to read 4 bytes but only got 0. 
  warnings.warn(str(msg))
/usr/local/lib/python3.7/dist-packages/PIL/TiffImagePlugin.py:788: UserWarning: Corrupt EXIF data.  Expecting to read 4 bytes but only got 0. 
  warnings.warn(str(msg))


epoch,train_loss,valid_loss,error_rate,time
0,0.545063,0.567544,0.170439,01:46
1,0.525474,0.639492,0.183755,01:45
2,0.494488,0.723573,0.183755,01:46
3,0.386943,0.525054,0.143808,01:45
4,0.267001,0.524209,0.135819,01:46
5,0.209896,0.526258,0.137150,01:44


/usr/local/lib/python3.7/dist-packages/PIL/TiffImagePlugin.py:788: UserWarning: Corrupt EXIF data.  Expecting to read 4 bytes but only got 0. 
  warnings.warn(str(msg))
/usr/local/lib/python3.7/dist-packages/PIL/TiffImagePlugin.py:788: UserWarning: Corrupt EXIF data.  Expecting to read 4 bytes but only got 0. 
  warnings.warn(str(msg))
/usr/local/lib/python3.7/dist-packages/PIL/TiffImagePlugin.py:788: UserWarning: Corrupt EXIF data.  Expecting to read 4 bytes but only got 0. 
  warnings.warn(str(msg))
/usr/local/lib/python3.7/dist-packages/PIL/TiffImagePlugin.py:788: UserWarning: Corrupt EXIF data.  Expecting to read 4 bytes but only got 0. 
  warnings.warn(str(msg))
/usr/local/lib/python3.7/dist-packages/PIL/TiffImagePlugin.py:788: UserWarning: Corrupt EXIF data.  Expecting to read 4 bytes but only got 0. 
  warnings.warn(str(msg))
/usr/local/lib/python3.7/dist-packages/PIL/TiffImagePlugin.py:788: UserWarning: Corrupt EXIF data.  Expecting to read 4 bytes but only got 0. 
  warnings.w

In [ ]:
interp = ClassificationInterpretation.from_learner(learn3)
interp.print_classification_report()

/usr/local/lib/python3.7/dist-packages/PIL/TiffImagePlugin.py:788: UserWarning: Corrupt EXIF data.  Expecting to read 4 bytes but only got 0. 
  warnings.warn(str(msg))


/usr/local/lib/python3.7/dist-packages/PIL/TiffImagePlugin.py:788: UserWarning: Corrupt EXIF data.  Expecting to read 4 bytes but only got 0. 
  warnings.warn(str(msg))


                          precision    recall  f1-score   support

            Acanthastrea       0.85      0.76      0.80        29
    Acropora Cervicornis       0.91      0.89      0.90        35
        Acropora Palmata       0.90      0.87      0.89        31
               Alveopora       0.95      0.73      0.83        26
             Blastomussa       0.80      0.84      0.82        19
  Catalaphyllia jardinae       0.95      0.90      0.92        20
     Caulastraea Furcata       1.00      0.95      0.97        20
              Cyphastrea       0.84      0.89      0.86        18
    Dendrogyra cylindrus       0.86      1.00      0.92        24
      Diadema Antillarum       1.00      1.00      1.00        21
       Diploria Strigosa       0.95      0.88      0.91        24
               Discosoma       0.91      0.78      0.84        27
          Duncanopsammia       0.96      1.00      0.98        25
               Euphyllia       0.81      0.78      0.79        27
        E

## Filtering using optical character recognition
###https://towardsdatascience.com/targeting-and-removing-bad-training-data-8ccdac5e7cc3
Tesseract is open-source software available for OCR that is straightforward to implement. Here, we use their image_to_string function to examine images after opening them with Pillow. As we’ll later need to look at entire folders of images, we add a simple clause that tells the function whether or not it needs to use Pillow to open the image, before returning the text (if any) that was found.

In [ ]:
!sudo apt install tesseract-ocr -q  &> /dev/null
!pip install pytesseract -q
import pytesseract

def optical_character_recognition(file, path = True):
  """ Simple OCR of text from images.

  Parameters
  ----------
  file: Path or str
    Image to examine.
  path: bool
    Indicates whether the file passed in is a path to a file, or an already opened Pillow Image.
    
  Returns:
  ----------
  str
    Text that was detected.
  """
  if path == True:
    # Use Pillow's Image class to open the image
    img = Image.open(file)
    new_size = tuple(2*x for x in img.size)
    img = img.resize(new_size, Image.ANTIALIAS)
    img = img.convert('L')
    text = pytesseract.image_to_string(img, lang='eng', config='-c tessedit_char_whitelist=01234567890ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz --psm 3 --oem 3')
    # Remove '\n\x0c' whiich is found in every image
    text = text[:-3]
  else: 
    # Already opened with Pillow
    new_size = tuple(2*x for x in file.size)
    img = file.resize(new_size, Image.ANTIALIAS)
    img = img.convert('L')
    text = pytesseract.image_to_string(file, lang='eng', config='-c tessedit_char_whitelist=01234567890ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz --psm 3 --oem 3')
    # Remove '\n\x0c' whiich is found in every image
    text = text[:-3]

  return text

In [ ]:
def crop_image(img_path, crop_pct = 0.1):
  """ Crops an image by a certain percent of the image size along the top and bottom of the image.

  Parameters
  ----------
  img_folder: Path or str
    Fold with images to examine.
  crop_pct: float (0<crop_pct<1)
    Percentage of the total height of the image to crop off top and bottom.
    
  Returns:
  ----------
  Cropped image (opened in PIL).
  """
  from PIL import Image
  img_file = Image.open(img_path).convert('RGB')
  [xs,ys] = img_file.size
  crop_x = crop_pct*xs
  crop_y = crop_pct*ys
  box = (0, crop_y, xs, ys-crop_y)
  cropped_image = img_file.crop(box)
  return cropped_image

In [ ]:

def find_artificial_text(img_folder, edge_removal = True, verbose = False):
  """ Searches for artificial text in each image in a image folder using pytesseract.

  Parameters
  ----------
  img_folder: Path or str
    Fold with images to examine.
  edge_removal: bool
    Should the function attempt to crop the edges of the to remove detected text.
    If no text is found, the cropped image is saved.
  verbose: bool
    Should the function print out which images are cropped, and which images have been found to have text.  
    
  Returns:
  ----------
  DataFrame containing images marked as possessing artifical text. 
  """
  from tqdm.notebook import tqdm
  # Initialize DataFrame
  column_names = ['Artificial Images', 'Text Found']
  df_mt = pd.DataFrame(columns = column_names)
  for image_path in tqdm(sorted(img_folder.ls())):
    im = Image.open(image_path).convert('RGB')
    if edge_removal == True:
      # if the image has text
      if len(optical_character_recognition(im, path = False))>0:
        if verbose == True:print(f'Text detected in image {image_path.name}');
        # if the text is along the top or bottom edge, overwrite the old image else make note of it in a dataframe
        if len(optical_character_recognition((crop_image(image_path)).convert('RGB'), path = False)) == 0:
          if verbose == True:print(f'Cropping removed text in image {image_path.name}');
          crop_image(image_path).save(image_path)
        else:
          df_mt = df_mt.append({'Artificial Images':image_path,'Text Found':optical_character_recognition(image_path)}, ignore_index=True)
    # No cropping      
    else:
      if len(optical_character_recognition(im, path=False))>0:
        if verbose==True:print(f'Text detected in image {image_path.name}');
        df_mt = df_mt.append({'Artificial Images':image_path,'Text Found':optical_character_recognition(image_path)}, ignore_index=True)

  # Create dataframe containing images with artificial text towards the center
  if verbose == True:print(f'{len(df_mt)} images with text found in folder {img_folder}.');
  return df_mt

In [ ]:
find_artificial_text(path/'Ctenella chagius')

In [ ]:
img = Image.open('/content/drive/MyDrive/corals/Ctenella chagius/00000132.jpg')
img